# 02 · Select, filter, CASE and sorting

**Scenario:** Transform real trip columns using Spark expressions rather than Python row loops.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Filter and derive columns

Column expressions are lazy. Use parentheses around each condition with & and |.


In [ ]:
valid=df.filter((F.col('distance')>0)&(F.col('fare')>=0))
enriched=(valid.withColumn('distance_band',F.when(F.col('distance')<2,'short').when(F.col('distance')<10,'medium').otherwise('long'))
    .withColumn('fare_per_mile',F.col('fare')/F.col('distance')))
assert enriched.filter('fare_per_mile IS NULL').count()==0
enriched.select('distance','fare','distance_band','fare_per_mile').show(8)


**Expected check:** Only positive-distance trips enter the division.


## Step 3 — Order results deterministically

A display limit is not a business ranking. Include a tie-breaker for deterministic top-N.


In [ ]:
top=enriched.orderBy(F.desc('fare'),F.asc('trip_id')).limit(10)
assert top.count()<=10
top.show(truncate=False)


**Expected check:** At most ten highest-fare rows appear, ordered by fare and fingerprint.


## Practice

Add a separate invalid band for null distance before the short/medium/long conditions.

## Solution guide

Place `when(col("distance").isNull(), "unknown")` first. CASE order matters; otherwise null comparisons fall through to otherwise.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
